# ALS Warm-Tail Diagnostics

This notebook investigates why the selected Implicit ALS model does not retrieve warm-tail movies in its top-10 recommendations.

It reproduces the frozen ALS reference from **05_final_hybrid_evaluation.ipynb**, then examines:

1. Item-factor norms against positive-interaction frequency.
2. The full-catalogue ALS rank and score gap of held-out warm-tail targets.
3. Whether the problem looks more like factor shrinkage or user-item ranking competition.

**This is a diagnostic notebook, not a model-tuning notebook.** The test window is used only to describe the frozen model's behaviour. Do not change hyperparameters or select a model based on these test diagnostics.



## Evaluation protocol and safeguards

- Positive interaction: rating **≥ 4.0**.
- Warm-tail item: **1–19 unique positive user-item interactions** in the data available before the test window.
- Strict cold-start item: **zero** positive pre-test interactions. Its ALS factor is not learned by this model, so it is excluded from ALS ranking.
- Model fitting: **train + validation**, matching the final pre-test refit in Notebook 05.
- Diagnostic targets: positive test interactions timestamped on or after **2016-01-01 00:00:00 UTC**.
- Candidate ranking: the full catalogue of items with learned ALS factors, excluding every item the user interacted with in train + validation (including ratings below 4).
- Tie-breaking: reproduce the deterministic tiny score perturbation used by the final hybrid evaluation, with tie seed 0.

The expected test slice in the current repository has approximately 26,658 relevant pairs overall and 1,810 warm-tail relevant pairs. Counts are printed from the local data rather than hard-coded, so the notebook also reveals when local data differs from the documented run.


In [ ]:
# Locate the repository root without relying on the notebook's working directory.
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

NAME = "sample_10pct"
K = 10
REL = 4.0
SEED = 42
CUTOFF_TS = int(
    pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp()
)

_START = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in (_START, *_START.parents):
    required = (
        candidate / "scripts" / "recutils.py",
        candidate / "data" / "processed" / NAME / "train.parquet",
    )
    if all(path.exists() for path in required):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Expected scripts/recutils.py "
        "and data/processed/sample_10pct/train.parquet. "
        "Run this notebook from the repository or one of its subfolders."
    )

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from recutils import load_split

print("Project root:", PROJECT_ROOT)
print("Dataset:", NAME)
print("Positive-rating threshold:", REL)
print("Test-window cutoff (UTC):", pd.to_datetime(CUTOFF_TS, unit="s", utc=True))


In [ ]:
# Load the frozen train / validation / test split and catalogue mappings.
S = load_split(NAME)

train = S["train"].copy()
val = S["val"].copy()
test = S["test"].copy()

user_map = S["user_map"].sort_values("u").reset_index(drop=True)
item_map = S["item_map"].sort_values("i").reset_index(drop=True)

n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

assert np.array_equal(user_map["u"].to_numpy(), np.arange(n_users))
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert item_map["i"].is_unique and item_map["movieId"].is_unique
assert user_map["u"].is_unique and user_map["userId"].is_unique

print(f"Users: {n_users:,} | Items: {n_items:,}")
print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(val):,}")
print(f"Test rows: {len(test):,}")


## 1. Define the pre-test history and warm-tail targets

The final hybrid notebook fits its test-evaluation ALS model on **train + validation**. We use the same data here so the factor norms, popularity buckets and target ranks describe the exact pre-test model.

Importantly, seen-item exclusion uses **all** train + validation interactions, while item frequency and warm-tail membership use only positive interactions (rating ≥ 4).


In [ ]:
# Construct positive item counts and the complete pre-test seen-item matrix.
train_val = pd.concat([train, val], ignore_index=True)

positive_tv = (
    train_val.loc[train_val["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
    .reset_index(drop=True)
)

n_pos_tv = np.bincount(
    positive_tv["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
)
n_all_interactions_tv = np.bincount(
    train_val["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
)

warm_tail_mask = (n_pos_tv >= 1) & (n_pos_tv <= 19)
strict_cold_mask = n_pos_tv == 0
item_has_positive_history = n_pos_tv > 0

u_has_pos = np.zeros(n_users, dtype=bool)
u_has_pos[positive_tv["u"].unique()] = True

seen_pairs_tv = train_val[["u", "i"]].drop_duplicates()
seen_tv = sp.csr_matrix(
    (
        np.ones(len(seen_pairs_tv), dtype=np.float32),
        (
            seen_pairs_tv["u"].to_numpy(dtype=np.int64),
            seen_pairs_tv["i"].to_numpy(dtype=np.int64),
        ),
    ),
    shape=(n_users, n_items),
)
seen_tv.sum_duplicates()
seen_tv.data[:] = 1.0

# Build the locked, post-2015 positive test target set.
test_ts = pd.to_numeric(test["timestamp"], errors="raise")
test_window = test.loc[
    (test["rating"] >= REL) & (test_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

target_u = test_window["u"].to_numpy(dtype=np.int64)
target_i = test_window["i"].to_numpy(dtype=np.int64)

# Test targets must not already be present in the pre-test interaction history.
assert not np.asarray(seen_tv[target_u, target_i]).ravel().any(), (
    "Found a post-cutoff test target in the pre-test seen-item matrix."
)

test_window["pretest_positive_count"] = n_pos_tv[target_i]
test_window["warm_tail"] = (
    (test_window["pretest_positive_count"] >= 1)
    & (test_window["pretest_positive_count"] <= 19)
)

warm_tail_test = test_window.loc[test_window["warm_tail"]].copy()

print(f"Pre-test positive user-item pairs: {len(positive_tv):,}")
print(f"Warm-tail catalogue items (1–19): {int(warm_tail_mask.sum()):,}")
print(f"Strict-cold catalogue items (0): {int(strict_cold_mask.sum()):,}")
print(f"Post-2015 positive test pairs: {len(test_window):,}")
print(
    f"Warm-tail test pairs: {len(warm_tail_test):,} "
    f"across {warm_tail_test['u'].nunique():,} users"
)
print(
    "Warm-tail test users with positive pre-test history:",
    int(u_has_pos[warm_tail_test["u"].unique()].sum()),
)

assert len(warm_tail_test) > 0, "No warm-tail targets found; check the split and cutoff."
assert u_has_pos[warm_tail_test["u"].unique()].all(), (
    "A warm-tail target user unexpectedly has no positive pre-test history."
)

display(
    pd.DataFrame(
        {
            "scope": ["Overall", "Warm-tail", "Strict-cold"],
            "relevant_pairs": [
                len(test_window),
                int(test_window["pretest_positive_count"].between(1, 19).sum()),
                int((test_window["pretest_positive_count"] == 0).sum()),
            ],
        }
    )
)


## 2. Reproduce the frozen Implicit ALS model

The configuration below is copied from the selected ALS reference in Notebook 05:

| Parameter | Value |
|---|---:|
| Positive interactions | Rating ≥ 4 |
| Latent factors | 64 |
| Alpha | 5 |
| Regularization λ | 80 |
| Iterations | 10 |
| Random seed | 42 |

The custom half-step is retained to match the repository implementation. Do not alter these values in this notebook.


In [ ]:
# Exact implicit-ALS update used by the repository's final hybrid evaluation.
def _implicit_half_step(M, Y, lam):
    """Solve one ALS half-step; M stores confidence-minus-one for positives."""
    n_rows = M.shape[0]
    k = Y.shape[1]
    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)

    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]
        if start == end:
            continue

        factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]

        A = YtY + (factors.T * confidence_minus_one) @ factors + reg
        b = factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)

    return out


def fit_implicit_als_frozen(
    df,
    k=64,
    alpha=5.0,
    lam=80.0,
    n_iter=10,
    seed=SEED,
):
    d = (
        df.loc[df["rating"] >= REL, ["u", "i"]]
        .drop_duplicates(["u", "i"])
    )

    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (
                d["u"].to_numpy(dtype=np.int64),
                d["i"].to_numpy(dtype=np.int64),
            ),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()

    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)

    start_time = time.time()
    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)

        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            elapsed = time.time() - start_time
            print(f"ALS iteration {iteration:02d}/{n_iter} | {elapsed:.1f}s")

    return {
        "P": P.astype(np.float32),
        "Q": Q.astype(np.float32),
    }


ALS_CONFIG = {
    "training_data": "train + validation",
    "positive_rating_threshold": REL,
    "factors": 64,
    "alpha": 5.0,
    "lambda": 80.0,
    "iterations": 10,
    "seed": SEED,
}

start = time.time()
als_model = fit_implicit_als_frozen(
    train_val,
    k=ALS_CONFIG["factors"],
    alpha=ALS_CONFIG["alpha"],
    lam=ALS_CONFIG["lambda"],
    n_iter=ALS_CONFIG["iterations"],
    seed=ALS_CONFIG["seed"],
)

P_als = als_model["P"]
Q_als = als_model["Q"]

assert P_als.shape == (n_users, ALS_CONFIG["factors"])
assert Q_als.shape == (n_items, ALS_CONFIG["factors"])
assert np.isfinite(P_als).all() and np.isfinite(Q_als).all()

item_has_als_factors = np.abs(Q_als).sum(axis=1) > 0
assert np.array_equal(item_has_als_factors, item_has_positive_history), (
    "Learned-item eligibility differs from the positive-history mask."
)

print(f"Frozen ALS fit complete in {(time.time() - start) / 60:.2f} minutes.")
print("User-factor matrix:", P_als.shape)
print("Item-factor matrix:", Q_als.shape)
print("Items with learned factors:", int(item_has_als_factors.sum()))
print("Items without learned factors:", int((~item_has_als_factors).sum()))


## 3. Item-factor norm diagnostics

For each item, calculate the L2 norm of its learned factor vector:

\[
\|q_i\|_2 = \sqrt{\sum_{f=1}^{64} q_{if}^2}
\]

We compare norms across positive-interaction buckets: 1–4, 5–9, 10–19, 20–49 and 50+.

**Interpretation caution:** A smaller item-factor norm is consistent with stronger shrinkage, but it does not prove that regularization is the cause. Matrix-factorization embeddings have scale and rotation ambiguities; norm comparisons are diagnostic evidence, not a causal test.


In [ ]:
# Build one row per catalogue item.
item_factor_norms = np.linalg.norm(Q_als, axis=1)

item_stats = item_map[["i", "movieId"]].copy()
item_stats["positive_count"] = n_pos_tv
item_stats["all_interaction_count"] = n_all_interactions_tv
item_stats["factor_norm"] = item_factor_norms
item_stats["has_learned_factor"] = item_has_als_factors

def popularity_band(count):
    if count == 0:
        return "0 (strict cold)"
    if count <= 4:
        return "1–4"
    if count <= 9:
        return "5–9"
    if count <= 19:
        return "10–19"
    if count <= 49:
        return "20–49"
    return "50+"

BAND_ORDER = ["0 (strict cold)", "1–4", "5–9", "10–19", "20–49", "50+"]
item_stats["popularity_band"] = pd.Categorical(
    [popularity_band(int(c)) for c in item_stats["positive_count"]],
    categories=BAND_ORDER,
    ordered=True,
)

# Attach titles when the repository's prepared movie metadata is available.
movies_path = PROJECT_ROOT / "data" / "interim" / NAME / "movies.parquet"
if movies_path.exists():
    movies = pd.read_parquet(movies_path)
    metadata_columns = [
        column for column in ["movieId", "title", "genres"]
        if column in movies.columns
    ]
    if "movieId" in metadata_columns:
        item_stats = item_stats.merge(
            movies[metadata_columns].drop_duplicates("movieId"),
            on="movieId",
            how="left",
        )
else:
    print("Optional movie titles unavailable; item IDs will be shown.")

factor_summary = (
    item_stats.groupby("popularity_band", observed=False)
    .agg(
        items=("i", "count"),
        median_positive_interactions=("positive_count", "median"),
        median_factor_norm=("factor_norm", "median"),
        q25_factor_norm=("factor_norm", lambda x: x.quantile(0.25)),
        q75_factor_norm=("factor_norm", lambda x: x.quantile(0.75)),
        mean_factor_norm=("factor_norm", "mean"),
    )
    .reset_index()
)

display(factor_summary.round(6))


In [ ]:
# Plot 1: factor norm distributions by positive-interaction bucket.
positive_items = item_stats.loc[item_stats["positive_count"] > 0].copy()

box_data = [
    positive_items.loc[
        positive_items["popularity_band"] == band, "factor_norm"
    ].to_numpy()
    for band in ["1–4", "5–9", "10–19", "20–49", "50+"]
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(
    box_data,
    labels=["1–4", "5–9", "10–19", "20–49", "50+"],
    showfliers=False,
)
ax.set_yscale("log")
ax.set_xlabel("Positive training interactions per item")
ax.set_ylabel("Item-factor L2 norm (log scale)")
ax.set_title("ALS item-factor norms by popularity bucket")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Plot 2: item-level norms against positive interaction counts.
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    positive_items["positive_count"],
    positive_items["factor_norm"],
    alpha=0.22,
    s=12,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Unique positive training interactions (log scale)")
ax.set_ylabel("Item-factor L2 norm (log scale)")
ax.set_title("Item-factor norm vs. positive interaction frequency")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4. Rank held-out warm-tail targets against the full catalogue

For each user with at least one warm-tail test target, we calculate the ALS score for every item:

\[
s(u,i) = p_u^\top q_i
\]

We then rank each held-out warm-tail target among all eligible items after:

- removing items with no learned ALS factor (strict cold-start);
- excluding every item already seen by that user in train + validation;
- applying the same deterministic tiny tie-break perturbation as Notebook 05.

The score is a latent ranking score, **not** a probability or predicted rating. We also calculate the score margin relative to the user's 10th-ranked eligible item.


In [ ]:
# Rank only the held-out warm-tail targets, scoring each target user's catalogue once.
TIE_SEED = 0
warm_targets_by_user = {
    int(user_id): group["i"].to_numpy(dtype=np.int64)
    for user_id, group in warm_tail_test.groupby("u", sort=True)
}

rank_rows = []
eligible_global = item_has_als_factors.copy()

for user_number, (user_id, target_items) in enumerate(warm_targets_by_user.items(), start=1):
    if not u_has_pos[user_id]:
        raise RuntimeError(f"Warm-tail target user {user_id} has no positive history.")

    # Raw full-catalogue ALS scores for this user.
    raw_scores = (
        P_als[user_id].astype(np.float32) @ Q_als.T
    ).astype(np.float64)

    # Match the ALS candidate eligibility and complete seen-item exclusion.
    raw_scores[~eligible_global] = -np.inf
    raw_scores[seen_tv[user_id].indices] = -np.inf

    eligible = np.isfinite(raw_scores)
    n_eligible = int(eligible.sum())
    if n_eligible < K:
        raise RuntimeError(
            f"User {user_id} has only {n_eligible} eligible ALS items."
        )

    # Match the deterministic tie-break used by final_top_k in Notebook 05.
    tie_rng = np.random.default_rng(
        np.random.SeedSequence([SEED, TIE_SEED, int(user_id)])
    )
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(raw_scores[eligible]))))

    adjusted_scores = raw_scores.copy()
    adjusted_scores[eligible] += tie_noise[eligible] * (1e-8 * scale)

    top10_cutoff = float(np.partition(adjusted_scores[eligible], -K)[-K])

    for item_id in target_items:
        if not eligible[item_id]:
            raise RuntimeError(
                f"Warm-tail target item {item_id} is not eligible for user {user_id}."
            )

        adjusted_target_score = float(adjusted_scores[item_id])
        target_rank = 1 + int(
            np.count_nonzero(adjusted_scores[eligible] > adjusted_target_score)
        )
        raw_target_score = float(raw_scores[item_id])

        rank_rows.append(
            {
                "u": int(user_id),
                "i": int(item_id),
                "pretest_positive_count": int(n_pos_tv[item_id]),
                "popularity_band": popularity_band(int(n_pos_tv[item_id])),
                "factor_norm": float(item_factor_norms[item_id]),
                "als_score": raw_target_score,
                "als_rank": target_rank,
                "eligible_catalogue_size": n_eligible,
                "rank_percentile": (
                    100.0 * (n_eligible - target_rank) / max(n_eligible - 1, 1)
                ),
                "score_gap_to_top10": adjusted_target_score - top10_cutoff,
            }
        )

    if user_number % 100 == 0 or user_number == len(warm_targets_by_user):
        print(
            f"Scored {user_number:,}/{len(warm_targets_by_user):,} "
            "warm-tail target users"
        )

rank_diagnostics = pd.DataFrame(rank_rows)

# Attach item identifiers and titles for inspection.
rank_diagnostics = rank_diagnostics.merge(
    item_stats.drop(columns=["positive_count", "all_interaction_count",
                             "factor_norm", "has_learned_factor",
                             "popularity_band"]),
    on="i",
    how="left",
)

rank_diagnostics["rank_bucket"] = pd.cut(
    rank_diagnostics["als_rank"],
    bins=[0, 10, 50, 100, 500, np.inf],
    labels=["1–10", "11–50", "51–100", "101–500", "500+"],
    right=True,
    ordered=True,
)

assert len(rank_diagnostics) == len(warm_tail_test)
assert rank_diagnostics["als_rank"].ge(1).all()

print(f"Ranked warm-tail test pairs: {len(rank_diagnostics):,}")
print(
    "Warm-tail targets in ALS top-10:",
    int(rank_diagnostics["als_rank"].le(10).sum()),
)


## 5. Summarize warm-tail ranking behaviour

Two summaries are useful:

- **Rank distribution:** where the warm-tail targets land in the full eligible catalogue.
- **Rank by interaction bucket:** whether the 1–4, 5–9 and 10–19 slices behave differently.

A negative score gap means the target's ALS score is below the user's 10th-item cutoff. A positive gap means it clears that cutoff.


In [ ]:
# Distribution of target ranks.
rank_bucket_order = ["1–10", "11–50", "51–100", "101–500", "500+"]

rank_distribution = (
    rank_diagnostics["rank_bucket"]
    .value_counts(sort=False)
    .reindex(rank_bucket_order, fill_value=0)
    .rename_axis("rank_bucket")
    .reset_index(name="test_pairs")
)
rank_distribution["share"] = (
    rank_distribution["test_pairs"] / len(rank_diagnostics)
)

display(rank_distribution.assign(share=lambda x: x["share"].map(lambda v: f"{v:.2%}")))

# Compare ranking behaviour across warm-tail popularity sub-buckets.
warm_band_order = ["1–4", "5–9", "10–19"]
rank_by_popularity = (
    rank_diagnostics.groupby("popularity_band", observed=False)
    .agg(
        test_pairs=("als_rank", "size"),
        median_rank=("als_rank", "median"),
        mean_rank=("als_rank", "mean"),
        top10_share=("als_rank", lambda x: x.le(10).mean()),
        top50_share=("als_rank", lambda x: x.le(50).mean()),
        top500_share=("als_rank", lambda x: x.le(500).mean()),
        median_score_gap_to_top10=("score_gap_to_top10", "median"),
        median_factor_norm=("factor_norm", "median"),
    )
    .reindex(warm_band_order)
    .reset_index()
)

display(rank_by_popularity.round(6))


In [ ]:
# Inspect examples: closest warm-tail targets and most suppressed targets.
display_columns = [
    column for column in [
        "u", "i", "movieId", "title", "genres",
        "pretest_positive_count", "factor_norm", "als_score",
        "als_rank", "score_gap_to_top10",
    ]
    if column in rank_diagnostics.columns
]

print("Best-ranked warm-tail targets:")
display(
    rank_diagnostics.sort_values(
        ["als_rank", "score_gap_to_top10"],
        ascending=[True, False],
    )[display_columns].head(15)
)

print("Most suppressed warm-tail targets:")
display(
    rank_diagnostics.sort_values(
        ["als_rank", "score_gap_to_top10"],
        ascending=[False, True],
    )[display_columns].head(15)
)


## 6. How to interpret the diagnostics

Use these patterns to decide what to investigate next. Do not treat any single plot as proof of causality.

| Observation | What it suggests | Follow-up |
|---|---|---|
| 1–19 interaction items have much smaller norms than head items | Possible tail-factor shrinkage | Inspect item-specific regularization and compare norms by exact count |
| Warm-tail norms overlap with head-item norms, but ranks remain very low | Norm magnitude alone does not explain the failure | Inspect user-item alignment and score competition |
| Many targets rank 11–50 or 51–100 | Model has some signal but top-10 cutoff is restrictive | Examine score margins and candidate reranking |
| Targets are mostly ranked 500+ | Weak collaborative signal or poor representation | Compare content scores and user-specific relevance |
| 1–4 items perform much worse than 10–19 items | Strong dependence on interaction support | Test count-aware confidence/regularization on validation |

**Next experiment decision:** only after reviewing these outputs should we decide whether to test item-specific confidence weighting, count-aware regularization, or a different ranking objective. Keep the deployed ALS bundle unchanged.


In [ ]:
# Export reproducible diagnostic tables (no model or deployment files are modified).
REPORT_DIR = PROJECT_ROOT / "reports" / "als_warm_tail_diagnostics"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

item_stats.to_csv(
    REPORT_DIR / "item_factor_diagnostics.csv",
    index=False,
)
rank_diagnostics.to_csv(
    REPORT_DIR / "warm_tail_target_ranks.csv",
    index=False,
)
factor_summary.to_csv(
    REPORT_DIR / "factor_norm_summary.csv",
    index=False,
)
rank_distribution.to_csv(
    REPORT_DIR / "warm_tail_rank_distribution.csv",
    index=False,
)
rank_by_popularity.to_csv(
    REPORT_DIR / "warm_tail_rank_by_popularity.csv",
    index=False,
)

run_config = {
    **ALS_CONFIG,
    "dataset": NAME,
    "evaluation_cutoff_utc": "2016-01-01T00:00:00Z",
    "warm_tail_definition": "1-19 unique positive pre-test user-item interactions",
    "seen_item_exclusion": "all train + validation interactions",
    "tie_seed": TIE_SEED,
    "test_used_for_tuning": False,
}
(REPORT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Diagnostics exported to:", REPORT_DIR)


## Diagnostic checklist

Before moving to a model change, record the answers:

1. Are warm-tail item-factor norms systematically lower than those of 20+ interaction items?
2. Does norm differ mainly between 1–4, 5–9 and 10–19 interactions, or is there substantial overlap?
3. Where do warm-tail test targets land in the full-catalogue ALS ranking?
4. Are they just outside top-10, or hundreds/thousands of positions away?
5. Does the score gap to the 10th item indicate a narrow miss or a large ranking deficit?

The answers should determine the next experiment. **Do not tune on the test ranks.** Use the same diagnostic logic on the validation window when selecting any change, and reserve the frozen test window for final assessment only.
